In [ ]:
import pandas as pd
import numpy as np
from datetime import timedelta
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix
import warnings
warnings.filterwarnings("ignore")

In [ ]:
users = pd.read_csv("/content/users.csv", parse_dates=["signup_date"])
plans = pd.read_csv("/content/plans.csv")
subscriptions = pd.read_csv("/content/subscriptions.csv",
                            parse_dates=["start_date", "end_date", "cancel_date"])
payments = pd.read_csv("/content/payments.csv", parse_dates=["payment_date"])
usage = pd.read_csv("/content/usage_daily.csv")
usage['usage_date'] = pd.to_datetime(usage['usage_date'], format='%d-%m-%Y')
campaigns = pd.read_csv("/content/campaign_touchpoints.csv",
                        parse_dates=["touch_date"])

In [ ]:

users.drop_duplicates(subset=["user_id"], inplace=True)
subscriptions.drop_duplicates(inplace=True)
payments.drop_duplicates(inplace=True)


users["city_tier"] = users["city_tier"]
plans["plan_name"] = plans["plan_name"]
payments["payment_status"] = payments["payment_status"]


users["city_tier"].fillna("unknown", inplace=True)
usage.fillna(0, inplace=True)

usage["is_outlier"] = usage["minutes_used"] > usage["minutes_used"].quantile(0.99)

In [ ]:
import os

output_dir = '../data'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

as_of_date = pd.to_datetime(usage["usage_date"].max())
users["tenure_days"] = (as_of_date - users["signup_date"]).dt.days
paid_payments = payments[payments["payment_status"] == "success"]
lifetime_paid = paid_payments.groupby("user_id").size().reset_index(name="lifetime_paid_months")

last_active = usage.groupby("user_id")["usage_date"].max().reset_index()
last_active.rename(columns={"usage_date": "last_active_date"}, inplace=True)

recent_4w = usage[usage["usage_date"] >= as_of_date - timedelta(weeks=4)]
engagement = recent_4w.groupby("user_id")["minutes_used"].sum().reset_index()

q1 = engagement["minutes_used"].quantile(0.33)
q2 = engagement["minutes_used"].quantile(0.66)

def engagement_band(x):
    if x <= q1:
        return "low"
    elif x <= q2:
        return "medium"
    else:
        return "high"

engagement["engagement_band"] = engagement["minutes_used"].apply(engagement_band)

dim_users = users.merge(lifetime_paid, on="user_id", how="left") \
                 .merge(last_active, on="user_id", how="left") \
                 .merge(engagement[["user_id", "engagement_band"]], on="user_id", how="left")

dim_users.fillna({"lifetime_paid_months":0, "engagement_band":"low"}, inplace=True)

dim_users.to_csv("../data/dim_users_enriched.csv", index=False)

In [ ]:
import os

output_dir = '../data'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

usage["week_start"] = usage["usage_date"] - pd.to_timedelta(usage["usage_date"].dt.weekday, unit='d')

weekly_usage = usage.groupby(["user_id", "week_start"]).agg(
    active_days_week=("usage_date", "nunique"),
    total_minutes_week=("minutes_used", "sum"),
    sessions_week=("sessions_count", "sum")
).reset_index()

weekly_payments = payments.groupby(["user_id",
                                    payments["payment_date"].dt.to_period("W").apply(lambda r: r.start_time)]
                                   ).agg(
    payment_attempts_week=("payment_status", "count"),
    payment_failures_week=("payment_status", lambda x: (x=="failed").sum())
).reset_index()

weekly_payments.columns = ["user_id","week_start",
                           "payment_attempts_week","payment_failures_week"]

fact_weekly = weekly_usage.merge(weekly_payments,
                                  on=["user_id","week_start"], how="left")

fact_weekly.fillna(0, inplace=True)
fact_weekly.to_csv("../data/fact_user_weekly.csv", index=False)

In [ ]:
import os

output_dir = '../data'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

window_start = as_of_date - timedelta(weeks=4)
model_window = fact_weekly[fact_weekly["week_start"] >= window_start]

agg_features = model_window.groupby("user_id").agg(
    active_days_4w=("active_days_week","sum"),
    total_minutes_4w=("total_minutes_week","sum"),
    sessions_4w=("sessions_week","sum"),
    payment_failures_4w=("payment_failures_week","sum")
).reset_index()


last_week = model_window.groupby("user_id").tail(1)
avg_prev = model_window.groupby("user_id")["active_days_week"].mean().reset_index()

trend = last_week.merge(avg_prev, on="user_id")
trend["usage_trend"] = trend["active_days_week_x"] - trend["active_days_week_y"]
trend = trend[["user_id","usage_trend"]]

model_data = agg_features.merge(trend, on="user_id", how="left") \
                         .merge(dim_users, on="user_id", how="left")


model_data["days_since_last_activity"] = (as_of_date - model_data["last_active_date"]).dt.days

future_cutoff = as_of_date + timedelta(days=14)

subscriptions["will_churn_14d"] = np.where(
    (subscriptions["cancel_date"].notna()) &
    (subscriptions["cancel_date"] <= future_cutoff),
    1, 0
)

churn_label = subscriptions.groupby("user_id")["will_churn_14d"].max().reset_index()

model_data = model_data.merge(churn_label, on="user_id", how="left")
model_data["will_churn_14d"].fillna(0, inplace=True)

model_data["as_of_date"] = as_of_date

model_data.to_csv("../data/model_churn_dataset.csv", index=False)

In [ ]:
features = [
    "active_days_4w",
    "total_minutes_4w",
    "sessions_4w",
    "payment_failures_4w",
    "usage_trend",
    "days_since_last_activity",
    "tenure_days",
    "lifetime_paid_months"
]

X = model_data[features]
y = model_data["will_churn_14d"]

split_index = int(len(model_data)*0.8)

X_train, X_test = X[:split_index], X[split_index:]
y_train, y_test = y[:split_index], y[split_index:]


scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

log_model = LogisticRegression()
log_model.fit(X_train_scaled, y_train)

log_preds = log_model.predict_proba(X_test_scaled)[:,1]
print("Logistic ROC-AUC:", roc_auc_score(y_test, log_preds))


rf_model = RandomForestClassifier(n_estimators=200, random_state=42)
rf_model.fit(X_train, y_train)

rf_preds = rf_model.predict_proba(X_test)[:,1]
print("RandomForest ROC-AUC:", roc_auc_score(y_test, rf_preds))

threshold = 0.4
rf_binary = (rf_preds >= threshold).astype(int)

print("\nConfusion Matrix:\n", confusion_matrix(y_test, rf_binary))
print("\nClassification Report:\n", classification_report(y_test, rf_binary))

importance = pd.DataFrame({
    "feature": features,
    "importance": rf_model.feature_importances_
}).sort_values(by="importance", ascending=False)

print("\nTop Churn Drivers:\n", importance.head(10))

print("\nPipeline Completed Successfully.")

Logistic ROC-AUC: 0.9197289972899729
RandomForest ROC-AUC: 0.9088482384823848

Confusion Matrix:
 [[362  48]
 [ 26  64]]

Classification Report:
               precision    recall  f1-score   support

           0       0.93      0.88      0.91       410
           1       0.57      0.71      0.63        90

    accuracy                           0.85       500
   macro avg       0.75      0.80      0.77       500
weighted avg       0.87      0.85      0.86       500


Top Churn Drivers:
                     feature  importance
1          total_minutes_4w    0.382021
2               sessions_4w    0.299207
6               tenure_days    0.289300
4               usage_trend    0.014748
0            active_days_4w    0.010772
7      lifetime_paid_months    0.003462
5  days_since_last_activity    0.000489
3       payment_failures_4w    0.000000

Pipeline Completed Successfully.
